In [90]:
import numpy as np
import importlib
import scipy.sparse as sp
import utils
import torch
import random
import trainer

importlib.reload(utils)
importlib.reload(trainer)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [91]:
adj, features, labels = utils.load_data("cora")
print("adj information")
print(type(adj), adj.shape)
print(adj.dtype)
print(type(adj.data), adj.data.shape)
print("feature information")
print(type(features), features.shape)
print(features.dtype)
print(np.unique(features).shape)
print(features.min(), features.max())
print("labels information")
print(type(labels), labels.shape)
print(labels.dtype)
print(labels.min(), labels.max())

adj information
<class 'scipy.sparse._coo.coo_matrix'> (2708, 2708)
float32
<class 'numpy.ndarray'> (5429,)
feature information
<class 'numpy.ndarray'> (2708, 1433)
float32
(2,)
0.0 1.0
labels information
<class 'numpy.ndarray'> (2708,)
int32
0 6


In [92]:
adj = adj + adj.T
adj[adj > 1] = 1
assert (adj != adj.T).nnz == 0 , "not undirected graph"
diag = adj.diagonal()
assert np.sum(diag!=0) == 0 , "have self_loop"

seed = 42
random.seed(seed)
np.random.seed(seed)

# 可以使用两种划分方式 parameter 1 or 2
idx_train, idx_val, idx_test = utils.split_data(labels, 2)
print(idx_train.shape, idx_val.shape, idx_test.shape)
adj = adj.tocoo()

(267,) (267,) (2174,)


In [93]:
print("adj information")
print(type(adj), adj.shape)
print(adj.dtype)
print(type(adj.data), adj.data.shape)
print("feature information")
print(type(features), features.shape)
print(features.dtype)
print(np.unique(features).shape)
print(features.min(), features.max())
print("labels information")
print(type(labels), labels.shape)
print(labels.dtype)
print(labels.min(), labels.max())

adj information
<class 'scipy.sparse._coo.coo_matrix'> (2708, 2708)
float32
<class 'numpy.ndarray'> (10556,)
feature information
<class 'numpy.ndarray'> (2708, 1433)
float32
(2,)
0.0 1.0
labels information
<class 'numpy.ndarray'> (2708,)
int32
0 6


In [101]:
module = trainer.victim_model(adj, features, labels,
                              idx_train, idx_val, idx_test,
                              device=device,
                              dropout=0.5, lr=0.01, weight_decay=5e-4,
                              model="GraphSAGE")

module.train(epochs=200)
pred = module.evalute()
print("acc=", np.sum(pred[idx_test] == labels[idx_test]) / len(idx_test))

<class 'torch.Tensor'>
acc= 0.827966881324747
